In [2]:
print(123)

123


In [4]:
#✅
from kafka import KafkaConsumer
import json
server = 'localhost:9092'
topic_name = 'rides'

In [5]:
#✅
from models import Ride, ride_deserializer

In [6]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    #1value_deserializer=ride_deserializer
    #2value_deserializer=lambda x: x
    #3value_deserializer=lambda x: json.loads(x)
    value_deserializer=ride_deserializer
)

In [1]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [ ]:
sql="""
CREATE TABLE processed_events_aggregated (
    window_start TIMESTAMP,
    PULocationID INTEGER,
    num_trips BIGINT,
    total_revenue DOUBLE PRECISION,
    PRIMARY KEY (window_start, PULocationID)
);
"""
cur.execute(sql)

In [ ]:
sql="""
CREATE TABLE processed_events_aggregated (
    window_start TIMESTAMP,
    PULocationID INTEGER,
    num_trips BIGINT,
    total_revenue DOUBLE PRECISION,
    PRIMARY KEY (window_start, PULocationID)
);
"""
cur.execute(sql)

In [9]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...


Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...


KeyboardInterrupt: 

In [ ]:
li = []
for record in consumer:
    li.append(record.value)
    print(record.value)

In [93]:
record = next(consumer)

KeyboardInterrupt: 

In [88]:
record.value

Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)

In [4]:
li = []
for record in consumer:
    li.append(record.value)
    print(record.value)

Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)
Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)
Ride(PULocationID=142, DOLocationID=237, trip_distance=2.28, total_amount=24.94, tpep_pickup_datetime=1761958147000)
Ride(PULocationID=163, DOLocationID=238, trip_distance=2.7, total_amount=25.62, tpep_pickup_datetime=1761955639000)
Ride(PULocationID=138, DOLocationID=261, trip_distance=12.87, total_amount=86.14, tpep_pickup_datetime=1761955200000)
Ride(PULocationID=138, DOLocationID=37, trip_distance=8.4, total_amount=48.65, tpep_pickup_datetime=1761956330000)
Ride(PULocationID=90, DOLocationID=100, trip_distance=0.85, total_amount=16.45, tpep_pickup_datetime=1761956471000)
Ride(PULocationID=142, DOLocationID=170, trip_distance=3.01, total_amount=25.85, tpep_pickup_datetime=1761955651000)
Ride(PULocationID=237, DOLocationID=144, trip_distance=3.82, total_am

KeyboardInterrupt: 

In [95]:
next(consumer)

KeyboardInterrupt: 

In [11]:
import dataclasses

topic_name = 'rides'

producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()

NameError: name 'producer' is not defined

In [ ]:
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [ ]:
producer.send(topic_name, value=ride)
producer.flush()

In [ ]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

In [6]:
sql="""
CREATE TABLE processed_events (
    PULocationID INTEGER,
    DOLocationID INTEGER,
    trip_distance DOUBLE PRECISION,
    total_amount DOUBLE PRECISION,
    pickup_datetime TIMESTAMP
);
"""
cur.execute(sql)

In [2]:
sql="""
CREATE TABLE processed_events_aggregated (
    window_start TIMESTAMP,
    PULocationID INTEGER,
    num_trips BIGINT,
    total_revenue DOUBLE PRECISION,
    PRIMARY KEY (window_start, PULocationID)
);
"""
cur.execute(sql)


In [19]:
sql="""
SELECT * FROM processed_events_aggregated;
"""
cur.execute(sql)
result = cur.fetchall()
result

[]

In [20]:
sql="""
SELECT window_start, count(*) as locations, sum(num_trips) as total_trips,
       round(sum(total_revenue)::numeric, 2) as revenue
FROM processed_events_aggregated
GROUP BY window_start
ORDER BY window_start;
"""
cur.execute(sql)
result = cur.fetchall()
for line in result:
    print(line)

In [130]:
sql="""
SELECT count(*) FROM processed_events_aggregated;
"""
cur.execute(sql)
result = cur.fetchall()
result

[(121,)]

In [137]:
sql="""
SELECT count(*) FROM processed_events;
"""
cur.execute(sql)
result = cur.fetchall()
result

[(2632,)]

In [106]:
sql="""
SELECT * FROM processed_events limit 500;
"""
cur.execute(sql)
result = cur.fetchall()
result

[(236, 186, 8.07, 52.29, datetime.datetime(2026, 5, 22, 20, 17, 54, 252000)),
 (237, 148, 11.3, 20.53, datetime.datetime(2026, 5, 22, 20, 17, 44, 274000)),
 (237, 230, 18.32, 40.4, datetime.datetime(2026, 5, 22, 20, 17, 54, 754000)),
 (48, 90, 9.22, 29.84, datetime.datetime(2026, 5, 22, 20, 17, 54, 775000)),
 (138, 164, 12.99, 79.27, datetime.datetime(2026, 5, 22, 20, 17, 55, 255000)),
 (249, 161, 15.44, 94.51, datetime.datetime(2026, 5, 22, 20, 17, 55, 275000)),
 (138, 138, 7.0, 96.96, datetime.datetime(2026, 5, 22, 20, 17, 55, 756000)),
 (107, 161, 19.42, 67.85, datetime.datetime(2026, 5, 22, 20, 17, 55, 776000)),
 (48, 234, 2.48, 63.58, datetime.datetime(2026, 5, 22, 20, 17, 56, 257000)),
 (237, 237, 10.22, 35.7, datetime.datetime(2026, 5, 22, 20, 17, 56, 277000)),
 (239, 162, 6.92, 11.4, datetime.datetime(2026, 5, 22, 20, 17, 56, 760000)),
 (230, 263, 6.43, 66.12, datetime.datetime(2026, 5, 22, 20, 17, 56, 778000)),
 (263, 148, 10.61, 47.17, datetime.datetime(2026, 5, 22, 20, 17, 5